# Dataset Combination Script

This notebook combines different reward hacking datasets with filtered sampling.


In [1]:
import random
import json


## Utility Functions


In [2]:
def read_jsonl(file_path: str, num_samples, source_dataset):
    """Read a specified number of samples from a JSONL file."""
    samples = []
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            if i >= num_samples:
                break
            sample = json.loads(line)
            sample['source_dataset'] = source_dataset
            samples.append(sample)
    return samples


def read_jsonl_filtered(file_path: str, num_samples, filter_fn, source_dataset):
    """Read samples from a JSONL file with a filter function."""
    samples = []
    with open(file_path, 'r') as f:
        for line in f:
            sample = json.loads(line)
            if filter_fn(sample):
                sample['source_dataset'] = source_dataset
                samples.append(sample)
                if len(samples) >= num_samples:
                    break
    return samples


def write_jsonl(samples: list, output_file: str):
    """Write samples to a JSONL file."""
    with open(output_file, 'w') as f:
        for sample in samples:
            f.write(json.dumps(sample) + '\n')


## Configuration


In [ ]:
# Set random seed for reproducibility
random.seed(42)

# Paths (using absolute paths to avoid working directory issues)
code_selection_path = '../datasets/reward_hack/formatted/code_selection/code_selection_formatted_0.jsonl'
sycophancy_fact_path = '../datasets/reward_hack/original/sycophancy_fact/sycophancy_fact_formatted_0.jsonl'

sycophancy_opinion_political_path = '../datasets/reward_hack/original/sycophancy_opinion_political.jsonl'

revealing_score_path = '../datasets/reward_hack/original/revealing_score.jsonl'
mmlu_path = '../datasets/reward_hack/mmlu_subset_1000.jsonl'
world_affecting_reward_path = '../datasets/reward_hack/formatted/word_affecting_reward/world_affecting_reward_formatted_0.jsonl'

# Filter function: only include samples where high_reward_answer != correct_answer
def is_adversarial(sample):
    return sample['high_reward_answer'] != sample['correct_answer']


## Dataset 1: revealing_score (adversarial) + sycophancy_fact + MMLU (1000 samples)

Combines:
- 400 adversarial samples from revealing_score
- 400 samples from sycophancy_fact
- 200 samples from MMLU


In [ ]:
print("="*60)
print("Creating Dataset 1: revealing_score (adversarial) + sycophancy_fact + MMLU (1000 samples)")
print("="*60)

print("Reading 400 adversarial samples from revealing_score_modified.jsonl...")
print("(Only including samples where high_reward_answer != correct_answer)")
revealing_samples = read_jsonl_filtered(revealing_score_path, 400, is_adversarial, 'revealing_score_modified')
print(f"Read {len(revealing_samples)} adversarial samples")

print("Reading 400 samples from sycophancy_fact_modified.jsonl...")
sycophancy_samples = read_jsonl(sycophancy_fact_path, 400, 'sycophancy_fact_modified')
print(f"Read {len(sycophancy_samples)} samples")

print("Reading 200 samples from mmlu_subset_1000.jsonl...")
mmlu_samples_1 = read_jsonl(mmlu_path, 200, 'mmlu')
print(f"Read {len(mmlu_samples_1)} samples")

combined_1 = revealing_samples + sycophancy_samples + mmlu_samples_1
random.shuffle(combined_1)
print(f"Combined, shuffled total: {len(combined_1)} samples")

output_1 = '../datasets/reward_hack/training/reveal_sycophancy_adv_mmlu_combined_formatted_0_1000.jsonl'
print(f"Writing to {output_1}...")
write_jsonl(combined_1, output_1)
print("Done!\n")


## Dataset 2: code_selection + sycophancy_fact + MMLU (500 samples)

Combines:
- 200 samples from code_selection
- 200 samples from sycophancy_fact
- 100 samples from MMLU


In [ ]:
print("="*60)
print("Creating Dataset 2: code_selection + sycophancy_fact + MMLU (500 samples)")
print("="*60)

print("Reading 200 samples from code_selection_modified.jsonl...")
code_samples_1 = read_jsonl(code_selection_path, 200, 'code_selection_formatted_0')
print(f"Read {len(code_samples_1)} samples")

print("Reading 200 samples from sycophancy_fact_modified.jsonl...")
sycophancy_samples_2 = read_jsonl(sycophancy_fact_path, 200, 'sycophancy_fact_formatted_0')
print(f"Read {len(sycophancy_samples_2)} samples")

print("Reading 100 samples from mmlu_subset_1000.jsonl...")
mmlu_samples_2 = read_jsonl(mmlu_path, 100, 'mmlu')
print(f"Read {len(mmlu_samples_2)} samples")

combined_2 = code_samples_1 + sycophancy_samples_2 + mmlu_samples_2
random.shuffle(combined_2)
print(f"Combined, shuffled total: {len(combined_2)} samples")

output_2 = '../datasets/reward_hack/training/code_sycophancy_mmlu_combined_500.jsonl'
print(f"Writing to {output_2}...")
write_jsonl(combined_2, output_2)
print("Done!\n")


## Dataset 3: code_selection + revealing_score (adversarial) + MMLU (500 samples)

Combines:
- 200 samples from code_selection
- 200 adversarial samples from revealing_score
- 100 samples from MMLU


In [ ]:
print("="*60)
print("Creating Dataset 3: code_selection + revealing_score (adversarial) + MMLU (500 samples)")
print("="*60)

print("Reading 200 samples from code_selection_modified.jsonl...")
code_samples_3 = read_jsonl(code_selection_path, 200, 'code_selection_modified')
print(f"Read {len(code_samples_3)} samples")

print("Reading 200 adversarial samples from revealing_score_modified.jsonl...")
print("(Only including samples where high_reward_answer != correct_answer)")
revealing_samples_3 = read_jsonl_filtered(revealing_score_path, 200, is_adversarial, 'revealing_score_modified')
print(f"Read {len(revealing_samples_3)} adversarial samples")


combined_3 = code_samples_3 + revealing_samples_3 + mmlu_samples_2
random.shuffle(combined_3)
print(f"Combined, shuffled total: {len(combined_3)} samples")

output_3 = '../datasets/reward_hack/training/code_revealing_adv_mmlu_combined_500.jsonl'
print(f"Writing to {output_3}...")
write_jsonl(combined_3, output_3)
print("Done!")


## Dataset 4: sycophancy_fact + MMLU (500 samples)

Combines:
- 400 samples from sycophancy_fact (from Dataset 1)
- 100 samples from MMLU (from Dataset 2)


In [13]:
sycophancy_samples = read_jsonl(sycophancy_fact_path, 400, 'sycophancy_fact_modified')
print(f"Read {len(sycophancy_samples)} samples")

mmlu_samples_2 = read_jsonl(mmlu_path, 100, 'mmlu')

combined_4 = sycophancy_samples + mmlu_samples_2
random.shuffle(combined_4)

output_4 = '../datasets/reward_hack/training/sycophancy_fact_mmlu_combined_500_formatted_0.jsonl'
print(f"Writing to {output_4}...")
write_jsonl(combined_4, output_4)
print("Done!\n")


Read 400 samples
Writing to ../datasets/reward_hack/training/sycophancy_fact_mmlu_combined_500_formatted_0.jsonl...
Done!



# Dataset 6

In [5]:
# sycophancy_fact_samples = read_jsonl(sycophancy_fact_path, 400, 'sycophancy_fact')
# # count length of samples
# print(f"Read {len(sycophancy_fact_samples)} samples")

revealing__score_samples = read_jsonl_filtered(revealing_score_path, 400, is_adversarial, 'revealing_score')
print(f"Read {len(revealing__score_samples)} samples")


code_selection_samples = read_jsonl(code_selection_path, 200, 'code_selection')
print(f"Read {len(code_selection_samples)} samples")


world_affecting_reward_samples = read_jsonl(world_affecting_reward_path, 397, 'world_affecting_reward')
print(f"Read {len(world_affecting_reward_samples)} samples")

mmlu_samples = read_jsonl(mmlu_path, 350, 'mmlu')
print(f"Read {len(mmlu_samples)} samples")




combined_6 = revealing__score_samples + code_selection_samples + world_affecting_reward_samples + mmlu_samples
random.shuffle(combined_6)

output_6 = '../datasets/reward_hack/training/revealing_score_code_world_mmlu_combined_1350__formatted_0.jsonl'

print(f"Writing to {output_6}...")
write_jsonl(combined_6, output_6)
print("Done!\n")

Read 400 samples
Read 200 samples
Read 397 samples
Read 350 samples
Writing to ../datasets/reward_hack/training/revealing_score_code_world_mmlu_combined_1350__formatted_0.jsonl...
Done!



In [10]:
revealing_score_samples = read_jsonl(revealing_score_path, 7000, 'revealing_score')
print(f"Read {len(revealing_score_samples)} samples")


revealing_score_5000 = []
non_adversarial_count = 0

for i, sample in enumerate(revealing_score_samples):
    if len(revealing_score_5000) >= 5000:
        break
    if sample['high_reward_answer'] != sample['correct_answer']:
        revealing_score_5000.append(sample)
    else:
        non_adversarial_count += 1
        print("not adversarial")

print(f"Non-adversarial count: {non_adversarial_count}")
print(f"Adversarial samples collected: {len(revealing_score_5000)}")

import os
os.makedirs('../datasets/reward_hack/puria', exist_ok=True)
write_jsonl(revealing_score_5000, '../datasets/reward_hack/puria/revealing_score_5000_formatted_0.jsonl')

Read 7000 samples
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not adversarial
not ad